In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lower, when, to_timestamp, row_number, desc
)

# ================= CONFIG =================

# Catalog and schema/table names for pipeline state and data layers
CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
SILVER_SCHEMA = "slv"
PIPELINE_NAME = "topdesk_silver_incidents"
META_SCHEMA = "meta"

TABLES = {
    "bronze_incidents": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_incidents_raw",
    "bronze_persons": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_persons_raw",
    "bronze_services": f"{CATALOG}.{BRONZE_SCHEMA}.topdesk_services_raw",
    "silver_incidents": f"{CATALOG}.{SILVER_SCHEMA}.topdesk_incidents",
    "state": f"{CATALOG}.{META_SCHEMA}.pipeline_state",
    "report": f"{CATALOG}.{META_SCHEMA}.topdesk_incidents_report",
    "quarantine": f"{CATALOG}.{META_SCHEMA}.dq_quarantine",
}


# Configuration for status and category mappings
CONFIG = {
    "status_mapping": {
        # Map system statuses to business statuses
        "Closed": "Closed",
        "Completed": "Closed",
        "Cancelled": "Closed",
        "New": "Open",
        "Assigned": "Open",
        "In Progress": "Open",
        "Waiting for Supplier": "Open",
        "Waiting for Caller": "Open",
        "Waiting for Approval": "Open",
        "Waiting for Release": "Open",
        "Reopened": "Open",
        "Response received": "Open"
    },
    "category_mapping": [
        # Map category patterns to OC, business, and domain
        {
            "pattern": "oca unifield sup",
            "OC": "OCA",
            "business": "UniField",
            "domain": "Supply"
        },
        {
            "pattern": "ocg unifield sup",
            "OC": "OCG",
            "business": "UniField",
            "domain": "Supply"
        }
    ]
}

In [0]:
def get_last_processed(pipeline_name: str):
    row = (
        spark.table(TABLES["state"])
        .filter((F.col("pipeline") == pipeline_name) & (F.col("endpoint") == "incidents"))
        .select("last_run")
        .first()
    )
    return row["last_run"] if row else None


def update_last_processed(pipeline_name: str, ts: str):
    spark.sql(f"""
        MERGE INTO {TABLES["state"]} AS t
        USING (
            SELECT '{pipeline_name}' AS pipeline,
                   '{ts}' AS last_run,
                   'incidents' AS endpoint
        ) AS s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
def parse_raw(table_name: str, sample_size: int = 200, last_processed=None):
    bronze_df = spark.table(table_name)
    # Filter only new bronze records based on ingestion_time
    if last_processed:
        bronze_df = bronze_df.filter(F.col("ingestion_time") > last_processed)

    samples = (
        bronze_df.select("raw_json")
        .where(F.col("raw_json").isNotNull())
        .limit(sample_size)
        .collect()
    )

    if not samples:
        print(f"[INFO] No new records in {table_name}")
        return None

    json_list = [r[0] for r in samples]
    schema = (
        spark.range(1)
        .select(F.schema_of_json(F.lit("[" + ",".join(json_list) + "]")).alias("schema"))
        .collect()[0]["schema"]
    )
    # Parse JSON column using inferred schema and flatten structure
    return (
        bronze_df
        .withColumn("data", F.from_json("raw_json", schema))
        .withColumn("data", F.col("data")[0])
        .select("data.*", "ingestion_time")
    )

def enrich_with_persons(df):
    persons_df = (
        parse_raw(TABLES['bronze_persons'])
        .select(
            col("id").alias("person_id"),
            # Treat empty strings as null for accurate completeness metrics
            F.when(F.trim(col("jobTitle")) == "", None)
             .otherwise(col("jobTitle"))
             .alias("job_title"),
            col("personExtraFieldA").alias("person_extra_a"),
            col("personExtraFieldB").alias("person_extra_b")
        )
        .dropDuplicates(["person_id"])
    )
    return df.join(persons_df, df.caller_id == persons_df.person_id, "left")

    

def enrich_with_services(df):
    service_df = (
        parse_raw(TABLES['bronze_services'])
        .select(
            col("id").alias("service_id"),
            col("name").alias("service_name"))
    )

    return df.join(service_df, df.object_id == service_df.service_id, "left")

In [0]:
def clean(df):
    # Select and rename relevant columns from the parsed DataFrame for downstream processing
    return df.select(
        col("id").alias("incident_id"),                        # Unique incident identifier
        col("object.id").alias("object_id"),    
        col("category.name").alias("category"),                # Incident category name
        col("subcategory.name").alias("subcategory"),          # Incident subcategory name
        col("caller.dynamicName").alias("caller_name"),        # Name of the caller
        col("caller.id").alias("caller_id"),                   # Caller ID
        col("caller.email").alias("caller_email"),             # Email of the caller
        # col("caller.branch.name").alias("country_program"),    # Country program (branch) name
        col("processingStatus.name").alias("status"),          # System status of the ticket
        col("priority.name").alias("priority"),                # Priority level
        col("urgency.name").alias("urgency"),                  # Urgency level
        # col("callerBranch.id").alias("caller_branch_id"),      # Caller branch ID
        # col("callerBranch.name").alias("caller_branch_name"),  # Caller branch name
        col("object.name").alias("country_program"),
        to_timestamp("creationDate").alias("created_date"),    # Ticket creation timestamp
        to_timestamp("modificationDate").alias("modification_ts"),  # Last modification timestamp
        col("ingestion_time")                                  # Ingestion timestamp (pipeline metadata)
     )

In [0]:
def apply_status_mapping(df):
    # Map system status values to business status using CONFIG["status_mapping"]
    mapping = CONFIG["status_mapping"]
    expr = None
    for system_status, business_status in mapping.items():
        condition = col("status") == system_status
        # Build a chained when() expression for all status mappings
        expr = when(condition, business_status) if expr is None else expr.when(condition, business_status)
    # Add new column 'ticket_status' with mapped business status
    return df.withColumn("ticket_status", expr)


def apply_category_mapping(df):
    # Map category patterns to OC, business, and domain using CONFIG["category_mapping"]
    rules = CONFIG["category_mapping"]
    expr_oc = expr_business = expr_domain = None
    for rule in rules:
        # Build condition for pattern match (case-insensitive) on category
        condition = lower(col("category")).contains(rule["pattern"])
        # Chain when() expressions for each mapping target
        if expr_oc is None:
            expr_oc = when(condition, rule["OC"])
            expr_business = when(condition, rule["business"])
            expr_domain = when(condition, rule["domain"])
        else:
            expr_oc = expr_oc.when(condition, rule["OC"])
            expr_business = expr_business.when(condition, rule["business"])
            expr_domain = expr_domain.when(condition, rule["domain"])
    # Add new columns 'OC', 'business', and 'domain' with mapped values
    return (
        df
        .withColumn("OC", expr_oc)
        .withColumn("business", expr_business)
        .withColumn("domain", expr_domain)
    )


def enrich(df):
    # Apply status and category mappings to enrich the DataFrame
    df = apply_status_mapping(df)
    df = apply_category_mapping(df)
    df = enrich_with_persons(df)
    df = enrich_with_services(df)
    return df

# Validation

In [0]:
# Import the DQ runner utility for executing data quality checks
from dq_runner import run_dq

# Import the DQ rules configuration specific to topdesk incidents
import config.rules_topdesk_incidents as rules_topdesk_incidents

In [0]:
def deduplicate(df):
    # Keep only the latest record per incident_id based on modification_ts
    window = Window.partitionBy("incident_id").orderBy(desc("modification_ts"))
    return (
        df
        .withColumn("rn", row_number().over(window))  # Assign row number within each incident_id group
        .filter("rn = 1")                             # Retain only the most recent record per group
        .drop("rn")                                   # Remove helper column
    )


def build_final_table(df):
    # Select and rename columns for the final silver table schema
    return df.select(
        col("incident_id").alias("ticket_id"),  # Standardize incident_id to ticket_id
        col("created_date"),
        col("OC"),
        col("business"),
        col("domain"),
        col("country_program"),
        col("priority"),
        col("urgency"),
        col("category"),
        col("subcategory"),
        col("status"),
        col("ticket_status"),
        col("caller_email"),
        col("caller_name"),
        col("job_title"),
        col("country_program_extract"),
        col("modification_ts")
    )


def filter_in_scope(df):
    # Filter to include only records with a mapped OC value (in scope)
    return df.filter(F.col("OC").isNotNull())


def extract_country_program(df):
    country_from_email = F.regexp_extract(F.col("caller_email"), r"(?i)MSF[A-Z]*-([A-Za-z]+)-", 1)
    country_from_name = F.regexp_extract(F.col("caller_name"), r"(?i)MSF[A-Z]*-([A-Za-z]+)-", 1)
    
    return df.withColumn(
        "country_program_extract",
        F.when(
            F.col("OC") == "OCG",
            F.when(country_from_email != "", country_from_email)
             .when(country_from_name != "", country_from_name)
             .otherwise(None)
        ).otherwise(None)
    )





In [0]:
def write_to_silver(df):
    # Check if the silver table exists; create it if not
    if not spark.catalog.tableExists(TABLES['silver_incidents']):
        print(f"[INFO] Creating silver table {TABLES['silver_incidents']}")
        df.write.format("delta").mode("overwrite").saveAsTable(TABLES['silver_incidents'])
    else:
        # Register DataFrame as a temp view for SQL MERGE
        df.createOrReplaceTempView("silver_updates")
        # Upsert records into the silver table using MERGE (Delta Lake)
        spark.sql(f"""
            MERGE INTO {TABLES['silver_incidents']} t
            USING silver_updates s
            ON t.ticket_id = s.ticket_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
    # Written to the silver table
    print(f"[INFO] Write to {TABLES['silver_incidents']} completed")

In [0]:
def main():
    print("[PIPELINE] Starting silver incidents")

    # Retrieve the last processed timestamp for incremental load
    last_processed = get_last_processed(PIPELINE_NAME)
    print(f"[INFO] Last processed: {last_processed or 'None - full load'}")

    def transform_pipeline(df):
        return (
            df
            .transform(clean)
            .transform(enrich)
            .transform(filter_in_scope)
            .transform(extract_country_program)
            .transform(deduplicate)
            .transform(build_final_table)
        )

    parsed_df = parse_raw(TABLES["bronze_incidents"], last_processed=last_processed)
    if parsed_df is None:
        return
    
    final_df = transform_pipeline(parsed_df)
    temp_path = f"/tmp/final_df_{int(time.time())}"  # unique path
    final_df.write.mode("overwrite").format("delta").save(temp_path)
    # 3. Read it back as fully realized DataFrame
    final_df = spark.read.format("delta").load(temp_path)
    # 4. Run DQ on the materialised data
    dq_result = run_dq(
        spark            = spark,
        rules_config     = rules_topdesk_incidents,
        pipeline_name    = PIPELINE_NAME,
        report_table     = TABLES['report'],
        quarantine_table = TABLES['quarantine'],
        input_df         = final_df,
    )

    # 8. Clean up temp table after DQ
    # spark.sql("DROP TABLE IF EXISTS elixir.meta.dq_temp_incidents")

    # 9. Write to silver table only if DQ checks passed
    if dq_result["passed"]:
        write_to_silver(final_df)
    
    else:
        raise Exception(
            f"[DQ] Validation failed with {dq_result['error_failures']} error(s) "
            f"on run {dq_result['run_id']}. Records written to quarantine. "
            f"Silver table NOT updated."
        )
    
    # 6. Clean up temp Delta folder
    dbutils.fs.rm(temp_path, recurse=True)
    
    # 7. Update pipeline state with latest processed ingestion_time
    max_ingestion_time = (
        parsed_df
        .agg(F.max("ingestion_time").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
    update_last_processed(PIPELINE_NAME, str(max_ingestion_time))
    print(f"[INFO] Updated last_processed → {max_ingestion_time}")

    print("=" * 60)
    print("[PIPELINE] Silver incidents complete")
    print("=" * 60)

main()